# Gravity Anomaly Mapping and Derivative Analysis
Gravity Method Tutorial #4

<b> Dicky Ahmad Zaky </b>
<br> Geophysical Engineering Department
<br> Universitas Pertamina

#### 1. Introduction

In regional and prospect-scale geophysics, gravity measurements are conducted across 2D grids $(x, y)$ or irregular station networks. From these data, geophysicists produce:
1. **Complete Bouguer Anomaly (CBA) Maps:** Visualizing lateral density variations across the survey block.
2. **First Horizontal Derivative (FHD / Total Horizontal Gradient THG):** Sharpens the lateral edges of anomalous bodies (faults, contacts, and basin boundaries):
   $$\text{FHD}(x, y) = \sqrt{\left(\frac{\partial g}{\partial x}\right)^2 + \left(\frac{\partial g}{\partial y}\right)^2}$$
   The peaks/maxima of FHD delineate structural discontinuities and lithological boundaries.

3. **Second Vertical Derivative (SVD):** Derived from Laplace's potential field equation in source-free space ($\nabla^2 g = \frac{\partial^2 g}{\partial x^2} + \frac{\partial^2 g}{\partial y^2} + \frac{\partial^2 g}{\partial z^2} = 0$):
   $$\text{SVD}(x, y) = \frac{\partial^2 g}{\partial z^2} = -\left(\frac{\partial^2 g}{\partial x^2} + \frac{\partial^2 g}{\partial y^2}\right)$$
   SVD acts as a high-pass spatial filter that resolves closely spaced bodies and isolates shallower residual targets. The **zero-contour line ($SVD = 0$)** corresponds to the vertical boundaries of geologic bodies.

#### 2. Synthetic 2D Survey Grid and Forward Model

We simulate a $5\text{ km} \times 5\text{ km}$ exploration block with grid spacing $\Delta x = \Delta y = 50\text{ m}$.
The model includes:
* **Target A (Intrusive Ore Body / Dense Stock):** Spherical body at $(x=1800\text{m}, y=2200\text{m})$, depth $z=300\text{m}$, radius $R=120\text{m}$, $\Delta\rho = +400\text{ kg/m}^3$.
* **Target B (Secondary Plug / Cavity):** Spherical body at $(x=3400\text{m}, y=3200\text{m})$, depth $z=250\text{m}$, radius $R=90\text{m}$, $\Delta\rho = +350\text{ kg/m}^3$.
* **Regional Tectonic Fault Step:** A diagonal fault block trending NW-SE producing a step-like density contrast across the survey area.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

# Gravitational constant
G = 6.67430e-11

# Survey Grid Parameters
x_min, x_max, dx = 0.0, 5000.0, 50.0
y_min, y_max, dy = 0.0, 5000.0, 50.0

x_coord = np.arange(x_min, x_max + dx, dx)
y_coord = np.arange(y_min, y_max + dy, dy)
X, Y = np.meshgrid(x_coord, y_coord)

# 1. Target A (Intrusive Stock)
xA, yA, zA = 1800.0, 2200.0, 300.0
rA, rhoA = 120.0, 400.0
vA = (4.0 / 3.0) * np.pi * (rA**3)
r_sq_A = (X - xA)**2 + (Y - yA)**2 + zA**2
g_targetA = (G * rhoA * vA * zA) / (r_sq_A**1.5) * 1e5  # mGal

# 2. Target B (Secondary Body)
xB, yB, zB = 3400.0, 3200.0, 250.0
rB, rhoB = 90.0, 350.0
vB = (4.0 / 3.0) * np.pi * (rB**3)
r_sq_B = (X - xB)**2 + (Y - yB)**2 + zB**2
g_targetB = (G * rhoB * vB * zB) / (r_sq_B**1.5) * 1e5  # mGal

# 3. Regional Fault Step (Sigmoid across line X + Y = 5000)
fault_axis = (X + Y - 5000.0) / 400.0
g_fault = 1.8 / (1.0 + np.exp(-fault_axis))  # Step anomaly across fault

# 4. Regional Monocline Trend
g_regional = 0.0003 * X + 0.0002 * Y + g_fault

# Total Complete Bouguer Anomaly (CBA) + Small Noise
np.random.seed(101)
noise = np.random.normal(0, 0.015, size=X.shape)
cba = g_targetA + g_targetB + g_regional + noise

print(f"Grid dimensions: {cba.shape[0]} x {cba.shape[1]} ({cba.size} stations)")
print(f"CBA range: {cba.min():.2f} to {cba.max():.2f} mGal")

#### 3. Computing First Horizontal Derivative (FHD) and Second Vertical Derivative (SVD)

We use numerical central differences (`np.gradient`) to compute 1st and 2nd spatial derivatives across the 2D grid.

In [ ]:
# 1. First Horizontal Derivatives
# np.gradient returns [dG/dy (along rows), dG/dx (along columns)]
dG_dy, dG_dx = np.gradient(cba, dy, dx)

# FHD / Total Horizontal Gradient (THG)
FHD = np.sqrt(dG_dx**2 + dG_dy**2) * 1000.0  # converted to mGal/km

# 2. Second Derivatives for SVD
d2G_dx2 = np.gradient(dG_dx, dx, axis=1)
d2G_dy2 = np.gradient(dG_dy, dy, axis=0)

# From Laplace's equation: SVD = -(d2G/dx2 + d2G/dy2)
SVD = -(d2G_dx2 + d2G_dy2) * 1e6  # converted to mGal/km^2

print(f"FHD max: {FHD.max():.2f} mGal/km")
print(f"SVD range: {SVD.min():.2f} to {SVD.max():.2f} mGal/km^2")

#### 4. Multi-Map Visualization: CBA, Shaded Relief, FHD, and SVD

Here we visualize the four diagnostic maps used by geophysicists for structural and anomaly interpretation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Complete Bouguer Anomaly (CBA) Map
im0 = axes[0, 0].contourf(X, Y, cba, levels=30, cmap='viridis')
c0 = axes[0, 0].contour(X, Y, cba, levels=12, colors='k', linewidths=0.5, alpha=0.5)
axes[0, 0].clabel(c0, inline=True, fontsize=8, fmt='%.1f')
axes[0, 0].plot([xA, xB], [yA, yB], 'r^', markersize=8, label='Source Centers')
axes[0, 0].set_title('(a) Complete Bouguer Anomaly (CBA) Map', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Easting X (m)')
axes[0, 0].set_ylabel('Northing Y (m)')
axes[0, 0].legend(loc='lower right')
cbar0 = fig.colorbar(im0, ax=axes[0, 0], orientation='vertical', fraction=0.046, pad=0.04)
cbar0.set_label('Bouguer Gravity (mGal)')

# 2. Shaded Relief Map (Illuminating structural boundaries)
ls = LightSource(azdeg=315, altdeg=45)
rgb = ls.shade(cba, cmap=plt.cm.terrain, vert_exag=0.5, blend_mode='overlay')
axes[0, 1].imshow(rgb, extent=(x_min, x_max, y_min, y_max), origin='lower')
axes[0, 1].set_title('(b) Shaded Relief / Hillshade Topography of Gravity', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Easting X (m)')
axes[0, 1].set_ylabel('Northing Y (m)')

# 3. First Horizontal Derivative (FHD) Map (Fault Boundary Delineation)
im2 = axes[1, 0].contourf(X, Y, FHD, levels=30, cmap='plasma')
axes[1, 0].set_title('(c) First Horizontal Derivative (FHD / Gradient)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Easting X (m)')
axes[1, 0].set_ylabel('Northing Y (m)')
cbar2 = fig.colorbar(im2, ax=axes[1, 0], orientation='vertical', fraction=0.046, pad=0.04)
cbar2.set_label('FHD (mGal/km)')

# 4. Second Vertical Derivative (SVD) Map with Zero Contour
im3 = axes[1, 1].contourf(X, Y, SVD, levels=30, cmap='coolwarm')
# Overlay zero contour (SVD = 0)
axes[1, 1].contour(X, Y, SVD, levels=[0], colors='black', linewidths=2.0, linestyles='-')
axes[1, 1].plot([xA, xB], [yA, yB], 'k^', markersize=8, label='Source Centers')
axes[1, 1].set_title('(d) Second Vertical Derivative (SVD) & Zero-Contour (black)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Easting X (m)')
axes[1, 1].set_ylabel('Northing Y (m)')
axes[1, 1].legend(loc='lower right')
cbar3 = fig.colorbar(im3, ax=axes[1, 1], orientation='vertical', fraction=0.046, pad=0.04)
cbar3.set_label('SVD (mGal/km$^2$)')

plt.tight_layout()
plt.show()

#### 5. Cross-Section Profile Analysis Across the Fault

To understand how the derivatives pinpoint geological contacts in cross-section, let's sample a profile cutting across the regional fault step (from $(x=0, y=5000)$ to $(x=5000, y=0)$).

In [ ]:
# Sample diagonal profile stations
diag_idx = np.arange(len(x_coord))
# Diagonal coordinates from top-left (0, 5000) to bottom-right (5000, 0)
x_prof = x_coord
y_prof = y_coord[::-1]

cba_prof = np.array([cba[len(y_coord) - 1 - i, i] for i in range(len(x_coord))])
fhd_prof = np.array([FHD[len(y_coord) - 1 - i, i] for i in range(len(x_coord))])
svd_prof = np.array([SVD[len(y_coord) - 1 - i, i] for i in range(len(x_coord))])
dist_prof = np.sqrt(x_prof**2 + (5000 - y_prof)**2)

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

ax1.plot(dist_prof, cba_prof, 'b-', linewidth=2)
ax1.set_ylabel('CBA (mGal)')
ax1.set_title('Profile Across Regional Fault Step', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--')

ax2.plot(dist_prof, fhd_prof, 'r-', linewidth=2)
fhd_max_idx = np.argmax(fhd_prof)
ax2.axvline(dist_prof[fhd_max_idx], color='k', linestyle='--', label=f'Fault Axis (Max FHD = {dist_prof[fhd_max_idx]:.0f} m)')
ax2.set_ylabel('FHD (mGal/km)')
ax2.legend()
ax2.grid(True, linestyle='--')

ax3.plot(dist_prof, svd_prof, 'm-', linewidth=2)
ax3.axhline(0, color='k', linestyle=':')
ax3.axvline(dist_prof[fhd_max_idx], color='k', linestyle='--', label='Zero-crossing at Fault')
ax3.set_xlabel('Profile Distance (m)')
ax3.set_ylabel('SVD (mGal/km$^2$)')
ax3.legend()
ax3.grid(True, linestyle='--')

plt.tight_layout()
plt.show()

#### 6. Exercises for Students

1. **Locating Structural Contacts:**
   * Look at Map (c) FHD and Map (d) SVD. Where does the maximum gradient peak in FHD lie relative to the zero-contour line in SVD? 
   * Explain mathematically why the peak of the first horizontal derivative corresponds to the inflection point ($\partial^2 g / \partial x^2 = 0$) in the gravity curve.

2. **Noise Vulnerability in Second Derivatives:**
   * Modify the noise level in the code from $\sigma = 0.015$ to $\sigma = 0.08\text{ mGal}$.
   * Recompute the SVD map. What happens to the zero-contour lines? Why are high-order derivatives hypersensitive to high-frequency noise?

3. **Field Practice Scenario:**
   * If you are tasked with placing exploratory drill holes to intersect the intrusive Target A and Target B, which map (CBA, FHD, or SVD) gives the most precise lateral center coordinates, and why?

In [ ]:
# Workspace for student exercise code